# 03 — Pré-processamento e Criação dos Datasets de Modelagem

Este notebook realiza o pré-processamento oficial da base meteorológica de São Paulo e cria os datasets finais para modelagem preditiva.

A base Parquet utilizada aqui **já está filtrada para o estado de São Paulo**.

Objetivos do notebook:

- limpar e padronizar os dados meteorológicos;
- tratar valores sentinela e valores fisicamente inválidos;
- preservar eventos climáticos extremos reais;
- criar variáveis temporais e geográficas;
- gerar uma base horária limpa;
- gerar uma base diária por estação;
- criar o dataset para prever a temperatura de amanhã;
- criar o dataset para prever a temperatura média dos próximos 7 dias;
- separar treino e teste temporalmente;
- aplicar imputação sem vazamento de dados;
- salvar todas as bases em Parquet.

Modelos esperados nas próximas etapas:

- Linear Regression;
- Random Forest Regressor;
- Redes Neurais.

## 1. Inicialização da SparkSession

A `SparkSession` é o ponto de entrada para trabalhar com Spark.

Como o projeto usa um dataset meteorológico grande, o processamento será feito com PySpark e consultas `spark.sql`.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Preprocessamento_Weather_SP_Regressao")
    .getOrCreate()
)

spark

## 2. Importação das bibliotecas

As transformações principais usam PySpark.

Este notebook não usa Pandas. Quando necessário, as bases são agregadas no Spark.

In [2]:
from pyspark.sql.functions import (
    col,
    when,
    count,
    avg,
    min,
    max,
    stddev,
    round,
    regexp_replace,
    regexp_extract,
    to_date,
    year,
    month,
    dayofmonth,
    hour,
    coalesce,
    sin,
    cos,
    lit,
    lead
)

from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.window import Window

from pyspark.ml.feature import Imputer, StringIndexer

import builtins
import unicodedata
import re
import math

## 3. Carregamento da base em Parquet

A base utilizada neste notebook vem do arquivo Parquet gerado no notebook anterior.

O formato Parquet é recomendado em projetos de Big Data porque armazena os dados em formato colunar, o que melhora a performance de leitura, compressão e seleção de colunas.

In [3]:
input_path = "/home/jovyan/work/data/processed/weather_sp_parquet"

df_raw = spark.read.parquet(input_path)

df_raw.createOrReplaceTempView("weather_raw")

spark.sql("""
    SELECT *
    FROM weather_raw
    LIMIT 5
""").show(truncate=False)

df_raw.printSchema()

+------+----------+-------------------+--------------------------------+-----------------------------------------------------+-----------------------------------------------+------------------------------------------------+-----------------------+--------------------------------------------+------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------------+------------------------------------------------+----------------------------------------+----------------------------------------+-----------------------------------+------------------------------------+--------------------------+-------------------------------+------+-----+--------+------------+------------+------------+------+
|index |Data      |Hora               |PRECIPITAÇÃO TOTAL, HORÁRIO (mm)|PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)|PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)|PRESSÃO ATMOSFERICA MIN. NA HO

## 4. Normalização dos nomes das colunas

Os nomes originais das colunas podem conter acentos, espaços, parênteses, vírgulas e outros caracteres especiais.

Para facilitar o uso das colunas em expressões Spark, todos os nomes são padronizados para:

- letras minúsculas;
- sem acentos;
- sem caracteres especiais;
- separação por `_`.

In [4]:
def normalizar_nome_coluna(nome):
    nome = nome.strip()
    nome = unicodedata.normalize("NFKD", nome)
    nome = nome.encode("ASCII", "ignore").decode("utf-8")
    nome = nome.lower()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    nome = re.sub(r"_+", "_", nome)
    nome = nome.strip("_")
    return nome

colunas_normalizadas = [normalizar_nome_coluna(c) for c in df_raw.columns]

df = df_raw.toDF(*colunas_normalizadas)

df.columns

['index',
 'data',
 'hora',
 'precipitacao_total_horario_mm',
 'pressao_atmosferica_ao_nivel_da_estacao_horaria_mb',
 'pressao_atmosferica_max_na_hora_ant_aut_mb',
 'pressao_atmosferica_min_na_hora_ant_aut_mb',
 'radiacao_global_kj_m2',
 'temperatura_do_ar_bulbo_seco_horaria_c',
 'temperatura_do_ponto_de_orvalho_c',
 'temperatura_maxima_na_hora_ant_aut_c',
 'temperatura_minima_na_hora_ant_aut_c',
 'temperatura_orvalho_max_na_hora_ant_aut_c',
 'temperatura_orvalho_min_na_hora_ant_aut_c',
 'umidade_rel_max_na_hora_ant_aut',
 'umidade_rel_min_na_hora_ant_aut',
 'umidade_relativa_do_ar_horaria',
 'vento_direcao_horaria_gr_gr',
 'vento_rajada_maxima_m_s',
 'vento_velocidade_horaria_m_s',
 'region',
 'state',
 'station',
 'station_code',
 'latitude',
 'longitude',
 'height']

## 5. Renomeação das variáveis principais

Após a normalização automática, algumas colunas ainda ficam com nomes muito longos.

Nesta etapa, as principais variáveis meteorológicas são renomeadas para nomes mais simples e legíveis. Isso melhora a leitura do código e facilita a etapa posterior de modelagem.

In [5]:
mapa_renomeacao = {
    "precipitacao_total_horario_mm": "precipitacao",
    "pressao_atmosferica_ao_nivel_da_estacao_horaria_mb": "pressao",
    "pressao_atmosferica_max_na_hora_ant_aut_mb": "pressao_maxima",
    "pressao_atmosferica_min_na_hora_ant_aut_mb": "pressao_minima",
    "radiacao_global_kj_m2": "radiacao",
    "temperatura_do_ar_bulbo_seco_horaria_c": "temperatura",
    "temperatura_do_ponto_de_orvalho_c": "temperatura_orvalho",
    "temperatura_maxima_na_hora_ant_aut_c": "temperatura_maxima",
    "temperatura_minima_na_hora_ant_aut_c": "temperatura_minima",
    "temperatura_orvalho_max_na_hora_ant_aut_c": "temperatura_orvalho_maxima",
    "temperatura_orvalho_min_na_hora_ant_aut_c": "temperatura_orvalho_minima",
    "umidade_relativa_do_ar_horaria": "umidade",
    "umidade_rel_max_na_hora_ant_aut": "umidade_maxima",
    "umidade_rel_min_na_hora_ant_aut": "umidade_minima",
    "vento_direcao_horaria_gr_gr": "direcao_vento",
    "vento_rajada_maxima_m_s": "rajada_vento",
    "vento_velocidade_horaria_m_s": "velocidade_vento",
    "height": "altitude"
}

for coluna_antiga, coluna_nova in mapa_renomeacao.items():
    if coluna_antiga in df.columns:
        df = df.withColumnRenamed(coluna_antiga, coluna_nova)

df.printSchema()

root
 |-- index: integer (nullable = true)
 |-- data: date (nullable = true)
 |-- hora: timestamp (nullable = true)
 |-- precipitacao: double (nullable = true)
 |-- pressao: double (nullable = true)
 |-- pressao_maxima: double (nullable = true)
 |-- pressao_minima: double (nullable = true)
 |-- radiacao: integer (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- temperatura_orvalho: double (nullable = true)
 |-- temperatura_maxima: double (nullable = true)
 |-- temperatura_minima: double (nullable = true)
 |-- temperatura_orvalho_maxima: double (nullable = true)
 |-- temperatura_orvalho_minima: double (nullable = true)
 |-- umidade_maxima: integer (nullable = true)
 |-- umidade_minima: integer (nullable = true)
 |-- umidade: integer (nullable = true)
 |-- direcao_vento: integer (nullable = true)
 |-- rajada_vento: double (nullable = true)
 |-- velocidade_vento: double (nullable = true)
 |-- region: string (nullable = true)
 |-- state: string (nullable = true)
 |-- statio

## 6. Seleção das colunas relevantes

Como o Parquet já representa São Paulo, o foco é manter:

- variáveis temporais;
- variáveis geográficas;
- identificadores de estação;
- variáveis meteorológicas.

In [7]:
colunas_uteis = [
    "data",
    "hora",
    "region",
    "state",
    "station",
    "station_code",
    "latitude",
    "longitude",
    "altitude",
    "temperatura",
    "temperatura_maxima",
    "temperatura_minima",
    "temperatura_orvalho",
    "temperatura_orvalho_maxima",
    "temperatura_orvalho_minima",
    "umidade",
    "umidade_maxima",
    "umidade_minima",
    "pressao",
    "pressao_maxima",
    "pressao_minima",
    "precipitacao",
    "radiacao",
    "velocidade_vento",
    "rajada_vento",
    "direcao_vento"
]

colunas_uteis = [c for c in colunas_uteis if c in df.columns]

df = df.select(*colunas_uteis)

df.show(5, truncate=False)
df.printSchema()

+----------+-------------------+------+-----+--------+------------+------------+------------+--------+-----------+------------------+------------------+-------------------+--------------------------+--------------------------+-------+--------------+--------------+-------+--------------+--------------+------------+--------+----------------+------------+-------------+
|data      |hora               |region|state|station |station_code|latitude    |longitude   |altitude|temperatura|temperatura_maxima|temperatura_minima|temperatura_orvalho|temperatura_orvalho_maxima|temperatura_orvalho_minima|umidade|umidade_maxima|umidade_minima|pressao|pressao_maxima|pressao_minima|precipitacao|radiacao|velocidade_vento|rajada_vento|direcao_vento|
+----------+-------------------+------+-----+--------+------------+------------+------------+--------+-----------+------------------+------------------+-------------------+--------------------------+--------------------------+-------+--------------+-------------

## 7. Conversão das colunas numéricas

As variáveis numéricas são convertidas para `double`.

Também é feita a troca de vírgula por ponto, caso algum valor decimal tenha vindo em formato brasileiro.

In [8]:
colunas_numericas = [
    "latitude",
    "longitude",
    "altitude",
    "temperatura",
    "temperatura_maxima",
    "temperatura_minima",
    "temperatura_orvalho",
    "temperatura_orvalho_maxima",
    "temperatura_orvalho_minima",
    "umidade",
    "umidade_maxima",
    "umidade_minima",
    "pressao",
    "pressao_maxima",
    "pressao_minima",
    "precipitacao",
    "radiacao",
    "velocidade_vento",
    "rajada_vento",
    "direcao_vento"
]

colunas_numericas = [c for c in colunas_numericas if c in df.columns]

for c in colunas_numericas:
    df = df.withColumn(
        c,
        regexp_replace(col(c).cast("string"), ",", ".").cast("double")
    )

df.printSchema()

root
 |-- data: date (nullable = true)
 |-- hora: timestamp (nullable = true)
 |-- region: string (nullable = true)
 |-- state: string (nullable = true)
 |-- station: string (nullable = true)
 |-- station_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- altitude: double (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- temperatura_maxima: double (nullable = true)
 |-- temperatura_minima: double (nullable = true)
 |-- temperatura_orvalho: double (nullable = true)
 |-- temperatura_orvalho_maxima: double (nullable = true)
 |-- temperatura_orvalho_minima: double (nullable = true)
 |-- umidade: double (nullable = true)
 |-- umidade_maxima: double (nullable = true)
 |-- umidade_minima: double (nullable = true)
 |-- pressao: double (nullable = true)
 |-- pressao_maxima: double (nullable = true)
 |-- pressao_minima: double (nullable = true)
 |-- precipitacao: double (nullable = true)
 |-- radiacao: double (null

## 8. Tratamento de valores sentinela

Valores como `-9999` e `-999` representam ausência de medição.

Eles serão convertidos para `null`.

In [9]:
for c in colunas_numericas:
    df = df.withColumn(
        c,
        when(col(c) <= -999, None).otherwise(col(c))
    )

df.show(5, truncate=False)

+----------+-------------------+------+-----+--------+------------+------------+------------+--------+-----------+------------------+------------------+-------------------+--------------------------+--------------------------+-------+--------------+--------------+-------+--------------+--------------+------------+--------+----------------+------------+-------------+
|data      |hora               |region|state|station |station_code|latitude    |longitude   |altitude|temperatura|temperatura_maxima|temperatura_minima|temperatura_orvalho|temperatura_orvalho_maxima|temperatura_orvalho_minima|umidade|umidade_maxima|umidade_minima|pressao|pressao_maxima|pressao_minima|precipitacao|radiacao|velocidade_vento|rajada_vento|direcao_vento|
+----------+-------------------+------+-----+--------+------------+------------+------------+--------+-----------+------------------+------------------+-------------------+--------------------------+--------------------------+-------+--------------+-------------

## 9. Criação das variáveis temporais

A data e a hora são transformadas em variáveis úteis.

Atenção: em alguns ambientes, a coluna `hora` pode estar como timestamp. Por isso, o código usa `hour(hora)` quando possível e só usa regex como alternativa.

In [10]:
df = df.withColumn(
    "data_formatada",
    when(
        to_date(col("data"), "yyyy-MM-dd").isNotNull(),
        to_date(col("data"), "yyyy-MM-dd")
    ).otherwise(
        to_date(col("data"), "dd/MM/yyyy")
    )
)

df = (
    df
    .withColumn("ano", year(col("data_formatada")))
    .withColumn("mes", month(col("data_formatada")))
    .withColumn("dia", dayofmonth(col("data_formatada")))
    .withColumn(
        "hora_num",
        coalesce(
            hour(col("hora")),
            regexp_extract(col("hora").cast("string"), r"(\d{2})", 1).cast("int")
        )
    )
)

df.select("data", "data_formatada", "ano", "mes", "dia", "hora", "hora_num").show(20, truncate=False)

+----------+--------------+----+---+---+-------------------+--------+
|data      |data_formatada|ano |mes|dia|hora               |hora_num|
+----------+--------------+----+---+---+-------------------+--------+
|2019-02-16|2019-02-16    |2019|2  |16 |2026-05-09 19:00:00|19      |
|2019-02-16|2019-02-16    |2019|2  |16 |2026-05-09 20:00:00|20      |
|2019-02-16|2019-02-16    |2019|2  |16 |2026-05-09 21:00:00|21      |
|2019-02-16|2019-02-16    |2019|2  |16 |2026-05-09 22:00:00|22      |
|2019-02-16|2019-02-16    |2019|2  |16 |2026-05-09 23:00:00|23      |
|2019-02-17|2019-02-17    |2019|2  |17 |2026-05-09 00:00:00|0       |
|2019-02-17|2019-02-17    |2019|2  |17 |2026-05-09 01:00:00|1       |
|2019-02-17|2019-02-17    |2019|2  |17 |2026-05-09 02:00:00|2       |
|2019-02-17|2019-02-17    |2019|2  |17 |2026-05-09 03:00:00|3       |
|2019-02-17|2019-02-17    |2019|2  |17 |2026-05-09 04:00:00|4       |
|2019-02-17|2019-02-17    |2019|2  |17 |2026-05-09 05:00:00|5       |
|2019-02-17|2019-02-

## 10. Criação de variáveis temporais cíclicas

Mês e hora têm comportamento cíclico.

Essas variáveis ajudam os modelos a entenderem que dezembro e janeiro estão próximos, assim como 23h e 0h.

In [11]:
df = (
    df
    .withColumn("mes_sin", sin(2 * lit(math.pi) * col("mes") / lit(12)))
    .withColumn("mes_cos", cos(2 * lit(math.pi) * col("mes") / lit(12)))
    .withColumn("hora_sin", sin(2 * lit(math.pi) * col("hora_num") / lit(24)))
    .withColumn("hora_cos", cos(2 * lit(math.pi) * col("hora_num") / lit(24)))
)

df.select("mes", "mes_sin", "mes_cos", "hora_num", "hora_sin", "hora_cos").show(20, truncate=False)

+---+------------------+------------------+--------+----------------------+---------------------+
|mes|mes_sin           |mes_cos           |hora_num|hora_sin              |hora_cos             |
+---+------------------+------------------+--------+----------------------+---------------------+
|2  |0.8660254037844386|0.5000000000000001|19      |-0.9659258262890684   |0.2588190451025203   |
|2  |0.8660254037844386|0.5000000000000001|20      |-0.8660254037844386   |0.5000000000000001   |
|2  |0.8660254037844386|0.5000000000000001|21      |-0.7071067811865477   |0.7071067811865474   |
|2  |0.8660254037844386|0.5000000000000001|22      |-0.5000000000000004   |0.8660254037844384   |
|2  |0.8660254037844386|0.5000000000000001|23      |-0.25881904510252157  |0.9659258262890681   |
|2  |0.8660254037844386|0.5000000000000001|0       |0.0                   |1.0                  |
|2  |0.8660254037844386|0.5000000000000001|1       |0.25881904510252074   |0.9659258262890683   |
|2  |0.8660254037844

## 11. Confirmação do recorte de São Paulo

Como o Parquet já está filtrado para SP, esta etapa apenas confirma o conteúdo da base.

In [12]:
df.createOrReplaceTempView("weather_sp_base")

spark.sql("""
    SELECT
        state,
        COUNT(*) AS total_registros,
        COUNT(DISTINCT station_code) AS total_estacoes,
        MIN(data_formatada) AS data_inicial,
        MAX(data_formatada) AS data_final
    FROM weather_sp_base
    GROUP BY state
    ORDER BY state
""").show(truncate=False)

+-----+---------------+--------------+------------+----------+
|state|total_registros|total_estacoes|data_inicial|data_final|
+-----+---------------+--------------+------------+----------+
|SP   |4288560        |43            |2001-08-30  |2021-04-30|
+-----+---------------+--------------+------------+----------+



## 12. Remoção de duplicatas

Como os dados são horários, espera-se uma medição por estação, data e hora.

In [13]:
linhas_antes = df.count()

colunas_chave = [c for c in ["station_code", "data_formatada", "hora_num"] if c in df.columns]

if len(colunas_chave) > 0:
    df = df.dropDuplicates(colunas_chave)
else:
    df = df.dropDuplicates()

linhas_depois = df.count()

print(f"Linhas antes: {linhas_antes:,}")
print(f"Linhas depois: {linhas_depois:,}")
print(f"Duplicatas removidas: {linhas_antes - linhas_depois:,}")

Linhas antes: 4,288,560
Linhas depois: 4,288,560
Duplicatas removidas: 0


## 13. Aplicação de regras físicas

Esta etapa remove apenas valores fisicamente inválidos.

Não será aplicada remoção genérica por IQR, pois eventos extremos podem representar fenômenos climáticos reais.

In [14]:
df_tratado = df

if "temperatura" in df_tratado.columns:
    df_tratado = df_tratado.filter(
        col("temperatura").isNull() |
        ((col("temperatura") >= -10) & (col("temperatura") <= 50))
    )

for c in ["temperatura_maxima", "temperatura_minima", "temperatura_orvalho", "temperatura_orvalho_maxima", "temperatura_orvalho_minima"]:
    if c in df_tratado.columns:
        df_tratado = df_tratado.filter(
            col(c).isNull() |
            ((col(c) >= -30) & (col(c) <= 50))
        )

for c in ["umidade", "umidade_maxima", "umidade_minima"]:
    if c in df_tratado.columns:
        df_tratado = df_tratado.filter(
            col(c).isNull() |
            ((col(c) >= 0) & (col(c) <= 100))
        )

for c in ["pressao", "pressao_maxima", "pressao_minima"]:
    if c in df_tratado.columns:
        df_tratado = df_tratado.filter(
            col(c).isNull() |
            ((col(c) >= 800) & (col(c) <= 1100))
        )

for c in ["precipitacao", "radiacao", "velocidade_vento", "rajada_vento"]:
    if c in df_tratado.columns:
        df_tratado = df_tratado.filter(
            col(c).isNull() |
            (col(c) >= 0)
        )

if "direcao_vento" in df_tratado.columns:
    df_tratado = df_tratado.filter(
        col("direcao_vento").isNull() |
        ((col("direcao_vento") >= 0) & (col("direcao_vento") <= 360))
    )

print(f"Antes das regras físicas: {df.count():,}")
print(f"Após regras físicas     : {df_tratado.count():,}")
print(f"Linhas removidas        : {df.count() - df_tratado.count():,}")

Antes das regras físicas: 4,288,560
Após regras físicas     : 4,278,809
Linhas removidas        : 9,751


## 14. Remoção de registros sem temperatura e sem tempo válido

Como os modelos são supervisionados e a temperatura é a variável central, registros sem temperatura não serão usados na criação dos datasets de previsão.

In [15]:
df_tratado = df_tratado.filter(col("temperatura").isNotNull())

df_tratado = df_tratado.filter(
    col("data_formatada").isNotNull() &
    col("ano").isNotNull() &
    col("mes").isNotNull() &
    col("hora_num").isNotNull()
)

print(f"Linhas após filtros essenciais: {df_tratado.count():,}")

Linhas após filtros essenciais: 3,849,112


## 15. Classificações geográficas

São criadas colunas para enriquecer análises e modelos:

- `macro_regiao_sp`;
- `tipo_area`;
- `faixa_altitude`.

As classificações são aproximadas e servem para análise exploratória/modelagem.

In [16]:
df_tratado = (
    df_tratado
    .withColumn(
        "faixa_altitude",
        when(col("altitude").isNull(), "sem_info")
        .when(col("altitude") < 300, "baixa_altitude")
        .when(col("altitude") < 700, "media_altitude")
        .otherwise("alta_altitude")
    )
    .withColumn(
        "macro_regiao_sp",
        when((col("longitude") <= -48.5) & (col("latitude") <= -23.5), "sul_sudoeste")
        .when((col("longitude") <= -48.5) & (col("latitude") > -23.5), "oeste_noroeste")
        .when((col("longitude") > -47.0) & (col("latitude") <= -23.5), "leste_sudeste")
        .when((col("longitude") > -47.0) & (col("latitude") > -23.5), "nordeste_vale")
        .otherwise("centro_metropolitana")
    )
    .withColumn(
        "tipo_area",
        when((col("altitude") < 100) & (col("longitude") > -47.0), "litoral")
        .when(col("altitude") >= 700, "serra_altitude")
        .when(
            col("station").rlike("(?i)SAO PAULO|SÃO PAULO|BARUERI|GUARULHOS|OSASCO|SANTO ANDRE|SANTO ANDRÉ|SAO BERNARDO|SÃO BERNARDO"),
            "urbano_metropolitano"
        )
        .otherwise("interior")
    )
)

df_tratado.createOrReplaceTempView("weather_sp_tratado")

spark.sql("""
    SELECT
        macro_regiao_sp,
        tipo_area,
        faixa_altitude,
        COUNT(*) AS total_registros,
        COUNT(DISTINCT station_code) AS total_estacoes
    FROM weather_sp_tratado
    GROUP BY macro_regiao_sp, tipo_area, faixa_altitude
    ORDER BY macro_regiao_sp, tipo_area, faixa_altitude
""").show(100, truncate=False)

+--------------------+--------------+--------------+---------------+--------------+
|macro_regiao_sp     |tipo_area     |faixa_altitude|total_registros|total_estacoes|
+--------------------+--------------+--------------+---------------+--------------+
|centro_metropolitana|interior      |baixa_altitude|146110         |2             |
|centro_metropolitana|interior      |media_altitude|688058         |8             |
|centro_metropolitana|serra_altitude|alta_altitude |381637         |4             |
|leste_sudeste       |litoral       |baixa_altitude|51787          |2             |
|leste_sudeste       |serra_altitude|alta_altitude |128414         |3             |
|nordeste_vale       |interior      |media_altitude|244872         |3             |
|nordeste_vale       |serra_altitude|alta_altitude |380143         |4             |
|oeste_noroeste      |interior      |media_altitude|1581021        |16            |
|oeste_noroeste      |serra_altitude|alta_altitude |124247         |2       

## 16. Percentual de nulos por coluna com Spark SQL

A tabela abaixo mostra o percentual de nulos após o tratamento de valores sentinela e regras físicas.

In [19]:
df_tratado.createOrReplaceTempView("weather_sp_tratado")

total_linhas = spark.sql("""
    SELECT COUNT(*) AS total
    FROM weather_sp_tratado
""").collect()[0]["total"]

expressoes_sql = []

for c in df_tratado.columns:
    expressoes_sql.append(f"""
        ROUND(
            SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) / {total_linhas} * 100,
            2
        ) AS `{c}`
    """)

query_nulos = f"""
    SELECT
        {",".join(expressoes_sql)}
    FROM weather_sp_tratado
"""

df_percentual_nulos = spark.sql(query_nulos)

df_percentual_nulos.show(truncate=False)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

## 17. Salvamento da base horária limpa

Esta base preserva a granularidade horária e pode ser usada para análises detalhadas.

In [ ]:
base_horaria_path = "/home/jovyan/work/data/processed/weather_sp_limpo_horario"

df_tratado.write.mode("overwrite").parquet(base_horaria_path)

print(f"Base horária limpa salva em: {base_horaria_path}")

## 18. Criação da base diária por estação

Para prever a temperatura de amanhã e da próxima semana, é melhor trabalhar com uma base diária.

Cada linha representa uma estação em um dia.

In [20]:
base_diaria = spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        ano,
        mes,
        latitude,
        longitude,
        altitude,
        macro_regiao_sp,
        tipo_area,
        faixa_altitude,

        ROUND(AVG(temperatura), 2) AS temp_media_dia,
        ROUND(MIN(temperatura), 2) AS temp_min_dia,
        ROUND(MAX(temperatura), 2) AS temp_max_dia,

        ROUND(AVG(temperatura_orvalho), 2) AS temp_orvalho_media_dia,

        ROUND(AVG(umidade), 2) AS umidade_media_dia,
        ROUND(MIN(umidade), 2) AS umidade_min_dia,
        ROUND(MAX(umidade), 2) AS umidade_max_dia,

        ROUND(AVG(pressao), 2) AS pressao_media_dia,
        ROUND(SUM(precipitacao), 2) AS precipitacao_total_dia,
        ROUND(AVG(radiacao), 2) AS radiacao_media_dia,
        ROUND(AVG(velocidade_vento), 2) AS vento_medio_dia,
        ROUND(MAX(rajada_vento), 2) AS rajada_max_dia,

        COUNT(temperatura) AS medicoes_temp_validas
    FROM weather_sp_tratado
    WHERE data_formatada IS NOT NULL
    GROUP BY
        station,
        station_code,
        data_formatada,
        ano,
        mes,
        latitude,
        longitude,
        altitude,
        macro_regiao_sp,
        tipo_area,
        faixa_altitude
    ORDER BY station_code, data_formatada
""")

base_diaria.createOrReplaceTempView("base_diaria_sp")

base_diaria.show(20, truncate=False)

print(f"Linhas da base diária: {base_diaria.count():,}")

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

## 19. Criação de features temporais diárias

São criadas variáveis cíclicas de mês e estatísticas móveis dos últimos dias.

Essas variáveis serão usadas para prever temperatura futura.

In [ ]:
janela_estacao_data = (
    Window
    .partitionBy("station_code")
    .orderBy("data_formatada")
)

janela_ultimos_3 = (
    Window
    .partitionBy("station_code")
    .orderBy("data_formatada")
    .rowsBetween(-2, 0)
)

janela_ultimos_7 = (
    Window
    .partitionBy("station_code")
    .orderBy("data_formatada")
    .rowsBetween(-6, 0)
)

janela_proximos_7 = (
    Window
    .partitionBy("station_code")
    .orderBy("data_formatada")
    .rowsBetween(1, 7)
)

base_diaria_features = (
    base_diaria
    .withColumn("mes_sin", sin(2 * lit(math.pi) * col("mes") / lit(12)))
    .withColumn("mes_cos", cos(2 * lit(math.pi) * col("mes") / lit(12)))
    .withColumn("temp_media_ontem", lead("temp_media_dia", -1).over(janela_estacao_data))
    .withColumn("temp_media_ultimos_3_dias", avg("temp_media_dia").over(janela_ultimos_3))
    .withColumn("temp_media_ultimos_7_dias", avg("temp_media_dia").over(janela_ultimos_7))
    .withColumn("umidade_media_ultimos_7_dias", avg("umidade_media_dia").over(janela_ultimos_7))
    .withColumn("precipitacao_ultimos_7_dias", spark_sum("precipitacao_total_dia").over(janela_ultimos_7))
)

base_diaria_features.createOrReplaceTempView("base_diaria_features")

base_diaria_features.show(20, truncate=False)

## 20. Criação do dataset para prever a temperatura de amanhã

O alvo será:

`temperatura_amanha`

Ele representa a temperatura média da mesma estação no dia seguinte.

In [ ]:
dataset_amanha = (
    base_diaria_features
    .withColumn("temperatura_amanha", lead("temp_media_dia", 1).over(janela_estacao_data))
    .filter(col("temperatura_amanha").isNotNull())
)

dataset_amanha.createOrReplaceTempView("dataset_amanha")

spark.sql("""
    SELECT
        COUNT(*) AS total_linhas,
        MIN(data_formatada) AS data_inicial,
        MAX(data_formatada) AS data_final,
        ROUND(AVG(temperatura_amanha), 2) AS media_alvo
    FROM dataset_amanha
""").show(truncate=False)

dataset_amanha.show(10, truncate=False)

## 21. Criação do dataset para prever a temperatura média dos próximos 7 dias

O alvo será:

`temperatura_media_proximos_7_dias`

Ele representa a média da temperatura média diária dos próximos 7 dias para a mesma estação.

In [ ]:
dataset_semana = (
    base_diaria_features
    .withColumn("temperatura_media_proximos_7_dias", avg("temp_media_dia").over(janela_proximos_7))
    .filter(col("temperatura_media_proximos_7_dias").isNotNull())
)

dataset_semana.createOrReplaceTempView("dataset_semana")

spark.sql("""
    SELECT
        COUNT(*) AS total_linhas,
        MIN(data_formatada) AS data_inicial,
        MAX(data_formatada) AS data_final,
        ROUND(AVG(temperatura_media_proximos_7_dias), 2) AS media_alvo
    FROM dataset_semana
""").show(truncate=False)

dataset_semana.show(10, truncate=False)

## 22. Separação temporal e imputação sem vazamento

A separação entre treino e teste será feita por ano.

A imputação será ajustada apenas no treino e aplicada no treino/teste.

As categorias geográficas serão indexadas usando `StringIndexer` ajustado somente no treino.

In [ ]:
features_numericas_base = [
    "ano",
    "mes_sin",
    "mes_cos",
    "latitude",
    "longitude",
    "altitude",

    "temp_media_dia",
    "temp_min_dia",
    "temp_max_dia",
    "temp_orvalho_media_dia",

    "umidade_media_dia",
    "umidade_min_dia",
    "umidade_max_dia",

    "pressao_media_dia",
    "precipitacao_total_dia",
    "radiacao_media_dia",
    "vento_medio_dia",
    "rajada_max_dia",

    "temp_media_ontem",
    "temp_media_ultimos_3_dias",
    "temp_media_ultimos_7_dias",
    "umidade_media_ultimos_7_dias",
    "precipitacao_ultimos_7_dias"
]

features_categoricas_base = [
    "macro_regiao_sp",
    "tipo_area",
    "faixa_altitude"
]

def preparar_dataset_modelagem(df_dataset, coluna_alvo, nome_dataset):
    print("=" * 80)
    print(f"Preparando dataset: {nome_dataset}")
    print(f"Alvo: {coluna_alvo}")

    features_numericas = [
        c for c in features_numericas_base
        if c in df_dataset.columns
    ]

    features_categoricas = [
        c for c in features_categoricas_base
        if c in df_dataset.columns
    ]

    colunas_base = [
        "station",
        "station_code",
        "data_formatada"
    ] + features_numericas + features_categoricas + [coluna_alvo]

    colunas_base = [c for c in colunas_base if c in df_dataset.columns]

    df_base = df_dataset.select(*colunas_base)

    df_base = df_base.filter(col(coluna_alvo).isNotNull())
    df_base = df_base.filter(col("ano").isNotNull())

    anos_disponiveis = [
        row["ano"]
        for row in df_base.select("ano")
            .distinct()
            .dropna()
            .orderBy("ano")
            .collect()
    ]

    qtd_anos = len(anos_disponiveis)
    qtd_anos_teste = builtins.max(1, int(qtd_anos * 0.2))

    anos_teste = anos_disponiveis[-qtd_anos_teste:]
    anos_treino = anos_disponiveis[:-qtd_anos_teste]

    print("Anos disponíveis:", anos_disponiveis)
    print("Anos de treino:", anos_treino)
    print("Anos de teste:", anos_teste)

    df_treino = df_base.filter(col("ano").isin(anos_treino))
    df_teste = df_base.filter(col("ano").isin(anos_teste))

    print(f"Treino antes da preparação: {df_treino.count():,}")
    print(f"Teste antes da preparação : {df_teste.count():,}")

    for c in features_categoricas:
        df_treino = df_treino.withColumn(c, when(col(c).isNull(), "sem_info").otherwise(col(c)))
        df_teste = df_teste.withColumn(c, when(col(c).isNull(), "sem_info").otherwise(col(c)))

    features_categoricas_indexadas = []

    for c in features_categoricas:
        coluna_indexada = f"{c}_idx"

        indexer = StringIndexer(
            inputCol=c,
            outputCol=coluna_indexada,
            handleInvalid="keep"
        )

        indexer_model = indexer.fit(df_treino)

        df_treino = indexer_model.transform(df_treino)
        df_teste = indexer_model.transform(df_teste)

        features_categoricas_indexadas.append(coluna_indexada)

    colunas_para_imputar = [
        c for c in features_numericas
        if c in df_treino.columns
    ]

    imputer = Imputer(
        inputCols=colunas_para_imputar,
        outputCols=[f"{c}_imputado" for c in colunas_para_imputar]
    ).setStrategy("median")

    imputer_model = imputer.fit(df_treino)

    df_treino = imputer_model.transform(df_treino)
    df_teste = imputer_model.transform(df_teste)

    features_numericas_imputadas = [f"{c}_imputado" for c in colunas_para_imputar]

    features_finais = features_numericas_imputadas + features_categoricas_indexadas

    colunas_saida = [
        "station",
        "station_code",
        "data_formatada"
    ] + features_finais + [coluna_alvo]

    df_treino_final = df_treino.select(*colunas_saida)
    df_teste_final = df_teste.select(*colunas_saida)
    df_completo_final = df_treino_final.unionByName(df_teste_final)

    print("Features finais:")
    for f in features_finais:
        print("-", f)

    print(f"Treino final: {df_treino_final.count():,}")
    print(f"Teste final : {df_teste_final.count():,}")
    print(f"Completo    : {df_completo_final.count():,}")

    return df_completo_final, df_treino_final, df_teste_final, features_finais

## 23. Preparação final do dataset de amanhã

In [ ]:
dataset_amanha_final, dataset_amanha_train, dataset_amanha_test, features_amanha = preparar_dataset_modelagem(
    df_dataset=dataset_amanha,
    coluna_alvo="temperatura_amanha",
    nome_dataset="previsao_amanha"
)

## 24. Preparação final do dataset da próxima semana

In [ ]:
dataset_semana_final, dataset_semana_train, dataset_semana_test, features_semana = preparar_dataset_modelagem(
    df_dataset=dataset_semana,
    coluna_alvo="temperatura_media_proximos_7_dias",
    nome_dataset="previsao_proximos_7_dias"
)

## 25. Verificação de nulos nos datasets finais

Os datasets finais devem estar sem nulos nas features e no alvo.

In [ ]:
def verificar_nulos(df_final, nome):
    print("=" * 80)
    print(nome)

    expressoes = [
        count(when(col(c).isNull(), 1)).alias(c)
        for c in df_final.columns
    ]

    df_final.select(*expressoes).show(truncate=False)

verificar_nulos(dataset_amanha_final, "Dataset amanhã completo")
verificar_nulos(dataset_amanha_train, "Dataset amanhã treino")
verificar_nulos(dataset_amanha_test, "Dataset amanhã teste")

verificar_nulos(dataset_semana_final, "Dataset semana completo")
verificar_nulos(dataset_semana_train, "Dataset semana treino")
verificar_nulos(dataset_semana_test, "Dataset semana teste")

## 26. Estatísticas finais dos alvos

Esta etapa valida se os alvos futuros ficaram coerentes.

In [ ]:
dataset_amanha_final.select(
    count("*").alias("total_registros"),
    round(avg("temperatura_amanha"), 2).alias("media_temperatura_amanha"),
    round(min("temperatura_amanha"), 2).alias("min_temperatura_amanha"),
    round(max("temperatura_amanha"), 2).alias("max_temperatura_amanha"),
    round(stddev("temperatura_amanha"), 2).alias("desvio_temperatura_amanha")
).show(truncate=False)

dataset_semana_final.select(
    count("*").alias("total_registros"),
    round(avg("temperatura_media_proximos_7_dias"), 2).alias("media_temperatura_semana"),
    round(min("temperatura_media_proximos_7_dias"), 2).alias("min_temperatura_semana"),
    round(max("temperatura_media_proximos_7_dias"), 2).alias("max_temperatura_semana"),
    round(stddev("temperatura_media_proximos_7_dias"), 2).alias("desvio_temperatura_semana")
).show(truncate=False)

## 27. Salvamento das bases finais

Serão salvas:

- base diária;
- dataset completo de amanhã;
- treino/teste de amanhã;
- dataset completo da próxima semana;
- treino/teste da próxima semana.

In [ ]:
base_diaria_path = "/home/jovyan/work/data/processed/weather_sp_base_diaria"

dataset_amanha_path = "/home/jovyan/work/data/processed/weather_sp_dataset_amanha"
dataset_amanha_train_path = "/home/jovyan/work/data/processed/weather_sp_amanha_train"
dataset_amanha_test_path = "/home/jovyan/work/data/processed/weather_sp_amanha_test"

dataset_semana_path = "/home/jovyan/work/data/processed/weather_sp_dataset_semana"
dataset_semana_train_path = "/home/jovyan/work/data/processed/weather_sp_semana_train"
dataset_semana_test_path = "/home/jovyan/work/data/processed/weather_sp_semana_test"

base_diaria.write.mode("overwrite").parquet(base_diaria_path)

dataset_amanha_final.write.mode("overwrite").parquet(dataset_amanha_path)
dataset_amanha_train.write.mode("overwrite").parquet(dataset_amanha_train_path)
dataset_amanha_test.write.mode("overwrite").parquet(dataset_amanha_test_path)

dataset_semana_final.write.mode("overwrite").parquet(dataset_semana_path)
dataset_semana_train.write.mode("overwrite").parquet(dataset_semana_train_path)
dataset_semana_test.write.mode("overwrite").parquet(dataset_semana_test_path)

print(f"Base diária salva em: {base_diaria_path}")

print(f"Dataset amanhã completo salvo em: {dataset_amanha_path}")
print(f"Dataset amanhã treino salvo em   : {dataset_amanha_train_path}")
print(f"Dataset amanhã teste salvo em    : {dataset_amanha_test_path}")

print(f"Dataset semana completo salvo em: {dataset_semana_path}")
print(f"Dataset semana treino salvo em   : {dataset_semana_train_path}")
print(f"Dataset semana teste salvo em    : {dataset_semana_test_path}")

## 28. Teste de leitura das bases salvas

Esta etapa confirma se os arquivos foram gravados corretamente.

In [ ]:
checks = {
    "base_diaria": base_diaria_path,
    "dataset_amanha": dataset_amanha_path,
    "amanha_train": dataset_amanha_train_path,
    "amanha_test": dataset_amanha_test_path,
    "dataset_semana": dataset_semana_path,
    "semana_train": dataset_semana_train_path,
    "semana_test": dataset_semana_test_path
}

for nome, caminho in checks.items():
    df_check = spark.read.parquet(caminho)
    print(f"{nome}: {df_check.count():,} linhas | {len(df_check.columns)} colunas")
    df_check.show(3, truncate=False)

## Conclusão do pré-processamento

Neste notebook, a base meteorológica de São Paulo foi preparada para modelagem preditiva.

Principais entregas:

- base horária limpa;
- base diária agregada por estação;
- dataset para prever a temperatura média de amanhã;
- dataset para prever a temperatura média dos próximos 7 dias;
- separação temporal entre treino e teste;
- imputação sem vazamento de dados;
- indexação de variáveis categóricas geográficas;
- salvamento das bases em Parquet.

Os próximos notebooks podem usar diretamente:

- `weather_sp_amanha_train`;
- `weather_sp_amanha_test`;
- `weather_sp_semana_train`;
- `weather_sp_semana_test`.

Essas bases serão usadas para treinar e comparar:

- Linear Regression;
- Random Forest Regressor;
- Redes Neurais.